In [1]:
!pip install langchain langchain-openai openai -q
!pip install ydata-profiling -q
!pip install plotly -q
!pip install gradio -q
!pip install fpdf2 -q
!pip install kaleido -q

print("All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 31.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.8 MB/s eta 0:00:00
All libraries installed!


In [2]:
!pip install kaleido==0.2.1 -q

print("Kaleido downgraded! Now restart the kernel...")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 23.9 MB/s eta 0:00:00:00:0100:01
Kaleido downgraded! Now restart the kernel...


In [3]:
import os
from kaggle_secrets import UserSecretsClient  
secrets = UserSecretsClient()
os.environ["NVIDIA_API_KEY"] = secrets.get_secret("NVIDIA_API_KEY")

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

print("API Key loaded!")

API Key loaded!


In [4]:
from openai import OpenAI

client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=os.environ["NVIDIA_API_KEY"]
)

response = client.chat.completions.create(
    model="meta/llama-3.1-8b-instruct",
    messages=[{"role": "user", "content": "Say: NVIDIA API connected on Kaggle!"}],
    max_tokens=50
)

print(response.choices[0].message.content)

**NVIDIA API Connected on Kaggle!**

**Success Message:**

API Connection Established
API Key Validated
NVIDIA API Ready for Use on Kaggle

**Example Use Case:**

```python
import kaggle
from k


In [5]:
import pandas as pd

df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
print(f"Dataset loaded! Shape: {df.shape}")
print(df.head())

Dataset loaded! Shape: (891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            37

In [6]:
import pandas as pd
import numpy as np

def profile_dataframe(df):
    """
    Takes a pandas DataFrame and returns a detailed
    text profile — this will be sent to the LLM as context.
    """
    profile = []

    # Basic Shape
    profile.append("DATASET OVERVIEW")
    profile.append(f"Total Rows    : {df.shape[0]}")
    profile.append(f"Total Columns : {df.shape[1]}")
    profile.append(f"Column Names  : {list(df.columns)}")

    # Column Data Types 
    profile.append("\nCOLUMN DATA TYPES")
    for col, dtype in df.dtypes.items():
        profile.append(f"  {col:20s} → {str(dtype)}")

    # Missing Value
    profile.append("\nMISSING VALUES")
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(2)
    for col in df.columns:
        if null_counts[col] > 0:
            profile.append(f"  {col:20s} → {null_counts[col]} missing ({null_pct[col]}%)")
    if null_counts.sum() == 0:
        profile.append("  No missing values found!")

    # Numeric Column Statistics
    profile.append("\nNUMERIC COLUMN STATISTICS")
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        stats = df[numeric_cols].describe().round(2)
        profile.append(stats.to_string())
    else:
        profile.append("  No numeric columns found.")

    # Categorical Column Insights
    profile.append("\nCATEGORICAL COLUMN INSIGHTS")
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in cat_cols:
        unique_vals = df[col].nunique()
        top_vals    = df[col].value_counts().head(5).to_dict()
        profile.append(f"\n  Column: {col}")
        profile.append(f"  Unique Values : {unique_vals}")
        profile.append(f"  Top 5 Values  : {top_vals}")

    # Duplicate Rows
    profile.append("\nDUPLICATE ROWS")
    dup_count = df.duplicated().sum()
    profile.append(f"  Duplicate Rows: {dup_count}")

    return "\n".join(profile)

df = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")

data_profile = profile_dataframe(df)
print(data_profile)

DATASET OVERVIEW
Total Rows    : 891
Total Columns : 12
Column Names  : ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

COLUMN DATA TYPES
  PassengerId          → int64
  Survived             → int64
  Pclass               → int64
  Name                 → object
  Sex                  → object
  Age                  → float64
  SibSp                → int64
  Parch                → int64
  Ticket               → object
  Fare                 → float64
  Cabin                → object
  Embarked             → object

MISSING VALUES
  Age                  → 177 missing (19.87%)
  Cabin                → 687 missing (77.1%)
  Embarked             → 2 missing (0.22%)

NUMERIC COLUMN STATISTICS
       PassengerId  Survived  Pclass     Age   SibSp   Parch    Fare
count       891.00    891.00  891.00  714.00  891.00  891.00  891.00
mean        446.00      0.38    2.31   29.70    0.52    0.38   32.20
std         257.35      0.4

In [7]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"]
)

def generate_llm_insights(data_profile: str, dataset_name: str = "Dataset") -> str:
    """
    Sends the data profile to NVIDIA LLM and returns
    detailed EDA insights in plain English.
    """

    prompt = f"""
You are an expert Data Scientist performing Exploratory Data Analysis (EDA).

Below is a complete statistical profile of the '{dataset_name}' dataset.
Analyze it carefully and provide insights.

{data_profile}

Please provide your analysis in the following structure:

1. DATASET SUMMARY
   - What kind of data is this?
   - What is the likely objective/use case?

2. DATA QUALITY ISSUES
   - Which columns have missing values and how serious is it?
   - Are there any columns that should be dropped?
   - Any data type issues?

3. KEY PATTERNS & INSIGHTS
   - What are the most interesting statistical observations?
   - Which numeric columns show high variance or skewness?
   - What do the categorical distributions tell us?

4. POTENTIAL CORRELATIONS TO INVESTIGATE
   - Which column pairs are likely correlated?
   - What relationships would you prioritize exploring?

5. ANOMALIES & OUTLIERS
   - Which columns likely contain outliers based on min/max vs mean?
   - How might these affect analysis?

Be specific, use the actual column names and numbers from the profile.
Write in clear, professional English as if presenting to a business team.
"""

    response = client.chat.completions.create(
        model="meta/llama-3.1-8b-instruct",
        messages=[
            {
                "role": "system",
                "content": "You are an expert data scientist who provides clear, accurate, and actionable EDA insights."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=1500,
        temperature=0.3  
    )

    return response.choices[0].message.content



print(" Sending data profile to LLM...\n")

llm_insights = generate_llm_insights(
    data_profile=data_profile,
    dataset_name="Titanic"
)

print("           LLM EDA INSIGHTS REPORT")
print(llm_insights)

 Sending data profile to LLM...

           LLM EDA INSIGHTS REPORT
**Titanic Dataset Analysis**

**1. DATASET SUMMARY**

This dataset appears to be a historical record of passengers who boarded the RMS Titanic, a British passenger liner that sank in the North Atlantic Ocean in 1912. The data includes demographic information, survival status, and other relevant details.

The likely objective of this dataset is to analyze the factors that contributed to the survival or demise of passengers during the tragic event. This could be useful for understanding the social and economic dynamics of the time, as well as informing modern-day disaster response and emergency preparedness efforts.

**2. DATA QUALITY ISSUES**

Upon reviewing the dataset, we notice the following data quality issues:

* Missing values: The columns with missing values are:
	+ Age (177 missing, 19.87% of total rows)
	+ Cabin (687 missing, 77.1% of total rows)
	+ Embarked (2 missing, 0.22% of total rows)
* Seriousness of mis

In [8]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

os.makedirs("/kaggle/working/charts", exist_ok=True)

def generate_visualizations(df: pd.DataFrame) -> list:
    """
    Automatically generates visualizations based on column types.
    Saves all charts as PNG images and returns list of file paths.
    """
    chart_paths = []

    # Drop useless columns for visualization
    df_viz = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"],
                     errors="ignore")

    numeric_cols     = df_viz.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_viz.select_dtypes(include=["object"]).columns.tolist()

    # Histograms for Numeric Columns
    print(" Generating histograms...")
    for col in numeric_cols:
        fig = px.histogram(
            df_viz,
            x=col,
            color="Survived" if "Survived" in df_viz.columns else None,
            title=f"Distribution of {col}",
            barmode="overlay",
            opacity=0.7,
            color_discrete_map={0: "#EF553B", 1: "#00CC96"}
        )
        fig.update_layout(
            template="plotly_white",
            title_font_size=16,
            legend_title="Survived (0=No, 1=Yes)"
        )
        path = f"/kaggle/working/charts/hist_{col}.png"
        fig.write_image(path)
        chart_paths.append(path)
        print(f"    Saved: hist_{col}.png")

    # Bar Charts for Categorical Columns
    print("\n Generating bar charts...")
    for col in categorical_cols:
        if "Survived" in df_viz.columns:
            survival_rate = df_viz.groupby(col)["Survived"].mean().reset_index()
            survival_rate.columns = [col, "Survival Rate"]
            fig = px.bar(
                survival_rate,
                x=col,
                y="Survival Rate",
                title=f"Survival Rate by {col}",
                color="Survival Rate",
                color_continuous_scale="RdYlGn",
                text_auto=".2f"
            )
        else:
            counts = df_viz[col].value_counts().reset_index()
            counts.columns = [col, "Count"]
            fig = px.bar(counts, x=col, y="Count",
                         title=f"Distribution of {col}")

        fig.update_layout(template="plotly_white", title_font_size=16)
        path = f"/kaggle/working/charts/bar_{col}.png"
        fig.write_image(path)
        chart_paths.append(path)
        print(f"    Saved: bar_{col}.png")

    # Correlation Heatmap
    print("\nGenerating correlation heatmap...")
    corr_matrix = df_viz[numeric_cols].corr().round(2)
    fig = px.imshow(
        corr_matrix,
        text_auto=True,
        color_continuous_scale="RdBu_r",
        title="Correlation Heatmap — Numeric Features",
        aspect="auto"
    )
    fig.update_layout(template="plotly_white", title_font_size=16)
    path = "/kaggle/working/charts/correlation_heatmap.png"
    fig.write_image(path)
    chart_paths.append(path)
    print(" Saved: correlation_heatmap.png")

    # Survival Count (Overall)
    print("\nGenerating survival overview chart...")
    if "Survived" in df_viz.columns:
        survival_counts = df_viz["Survived"].value_counts().reset_index()
        survival_counts.columns = ["Survived", "Count"]
        survival_counts["Label"] = survival_counts["Survived"].map(
            {0: "Did Not Survive", 1: "Survived"}
        )
        fig = px.pie(
            survival_counts,
            values="Count",
            names="Label",
            title="Overall Survival Distribution",
            color_discrete_sequence=["#EF553B", "#00CC96"]
        )
        fig.update_layout(title_font_size=16)
        path = "/kaggle/working/charts/survival_overview.png"
        fig.write_image(path)
        chart_paths.append(path)
        print("  Saved: survival_overview.png")

    print(f"\nTotal charts generated: {len(chart_paths)}")
    return chart_paths


chart_paths = generate_visualizations(df)
print("\nChart paths:", chart_paths)

 Generating histograms...
    Saved: hist_Survived.png
    Saved: hist_Pclass.png
    Saved: hist_Age.png
    Saved: hist_SibSp.png
    Saved: hist_Parch.png
    Saved: hist_Fare.png

 Generating bar charts...
    Saved: bar_Sex.png
    Saved: bar_Embarked.png

Generating correlation heatmap...
 Saved: correlation_heatmap.png

Generating survival overview chart...
  Saved: survival_overview.png

Total charts generated: 10

Chart paths: ['/kaggle/working/charts/hist_Survived.png', '/kaggle/working/charts/hist_Pclass.png', '/kaggle/working/charts/hist_Age.png', '/kaggle/working/charts/hist_SibSp.png', '/kaggle/working/charts/hist_Parch.png', '/kaggle/working/charts/hist_Fare.png', '/kaggle/working/charts/bar_Sex.png', '/kaggle/working/charts/bar_Embarked.png', '/kaggle/working/charts/correlation_heatmap.png', '/kaggle/working/charts/survival_overview.png']


In [9]:
def generate_hypotheses_v2(data_profile: str, 
                           llm_insights: str,
                           dataset_name: str = "Dataset") -> str:
    """
    Improved hypothesis generator with:
    - Domain context injection
    - Stricter reasoning requirements
    - Proxy variable awareness
    - Confounding variable detection
    """

    prompt = f"""
You are a senior Data Scientist with deep domain knowledge 
analyzing the '{dataset_name}' dataset.

DOMAIN CONTEXT FOR TITANIC:
- This is a life-or-death survival dataset from 1912
- Key survival factors historically: sex, age, class, fare
- 'Embarked' is likely a PROXY variable (C=Cherbourg had 
   wealthier passengers → 1st class → higher survival)
- 'Pclass' directly reflects socioeconomic status
- 'Fare' and 'Pclass' are likely correlated (multicollinearity)
- "Women and children first" was the actual evacuation policy
- 3rd class passengers had restricted access to lifeboats

DATASET PROFILE:
{data_profile}

INITIAL INSIGHTS:
{llm_insights}

STRICT REQUIREMENTS FOR EACH HYPOTHESIS:
1. Must reference ACTUAL column names and statistics
2. Must identify if variable is direct cause or proxy
3. Must suggest the RIGHT statistical test for that data type
4. Must consider confounding variables
5. Must quantify expected business impact

Generate exactly 5 hypotheses in this format:

HYPOTHESIS [N]:
- CLAIM          : Specific, measurable, testable statement 
                   using actual column names
- TYPE           : Direct Effect / Proxy Variable / 
                   Confounded Relationship
- REASONING      : Data-driven reasoning using actual 
                   stats from the profile
- CONFOUNDERS    : Which other variables might explain 
                   this relationship
- HOW TO TEST    : Exact statistical test + Python library
- EXPECTED RESULT: Quantified expected outcome
- BUSINESS IMPACT: Real-world decision this enables

Focus on the CAUSAL story, not just correlation.
"""

    response = client.chat.completions.create(
        model="meta/llama-3.1-8b-instruct",
        messages=[
            {
                "role": "system",
                "content": """You are a senior data scientist who:
- Distinguishes correlation from causation
- Identifies proxy variables and confounders  
- Recommends precise statistical tests
- Thinks about real business impact
- Uses actual data statistics in reasoning"""
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_tokens=2500,
        temperature=0.3  
    )

    return response.choices[0].message.content


# Run improved version
print("Generating hypotheses with domain context...\n")

hypotheses_v2 = generate_hypotheses_v2(
    data_profile = data_profile,
    llm_insights = llm_insights,
    dataset_name = "Titanic"
)

print("     5 HYPOTHESES (with Causal Reasoning)")
print(hypotheses_v2)

hypotheses = hypotheses_v2
print("\nHypotheses updated! PDF will use improved version.")

Generating hypotheses with domain context...

     5 HYPOTHESES (with Causal Reasoning)
HYPOTHESIS 1:
- CLAIM          : Passengers in higher socioeconomic classes (Pclass) have a higher survival rate.
- TYPE           : Direct Effect
- REASONING      : The mean Pclass is 2.31, with a standard deviation of 0.84. This suggests a relatively balanced distribution of passenger classes. The mean survival rate is 38%, indicating that a significant proportion of passengers did not survive. However, passengers in higher socioeconomic classes (Pclass 1) have a significantly higher survival rate (expected result: 60% survival rate).
- CONFOUNDERS    : Sex, Age, Fare, Embarked
- HOW TO TEST    : Use a logistic regression model with Pclass as the primary predictor variable and Survived as the response variable. Use Python library statsmodels for logistic regression.
- EXPECTED RESULT: A statistically significant positive relationship between Pclass and survival rate.
- BUSINESS IMPACT: This enable

In [10]:
from fpdf import FPDF
import os
from datetime import datetime

class EDAReportPDF(FPDF):
    """Custom PDF class with header and footer."""

    def header(self):
        self.set_font("Arial", "B", 12)
        self.set_fill_color(30, 30, 30)
        self.set_text_color(255, 255, 255)
        self.cell(0, 10, "  LLM-Powered Automated EDA Report", 
                  fill=True, ln=True)
        self.set_text_color(0, 0, 0)
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font("Arial", "I", 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f"Page {self.page_no()} | Generated on "
                  f"{datetime.now().strftime('%Y-%m-%d %H:%M')}", 
                  align="C")

    def section_title(self, title: str):
        """Add a styled section title."""
        self.set_font("Arial", "B", 13)
        self.set_fill_color(240, 240, 240)
        self.set_text_color(30, 30, 30)
        self.cell(0, 9, f"  {title}", fill=True, ln=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body_text(self, text: str):
        """Add clean body text, handling encoding issues."""
        self.set_font("Arial", "", 10)
        # Remove characters that fpdf can't handle
        clean = text.encode("latin-1", "replace").decode("latin-1")
        self.multi_cell(0, 6, clean)
        self.ln(2)


def generate_pdf_report(dataset_name: str,
                        data_profile: str,
                        llm_insights: str,
                        hypotheses: str,
                        chart_paths: list) -> str:
    """
    Generates a complete EDA PDF report and returns the file path.
    """

    pdf = EDAReportPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # Cover Info
    pdf.set_font("Arial", "B", 20)
    pdf.set_text_color(30, 30, 30)
    pdf.ln(5)
    pdf.cell(0, 12, f"{dataset_name} Dataset", ln=True, align="C")
    pdf.set_font("Arial", "", 11)
    pdf.set_text_color(100, 100, 100)
    pdf.cell(0, 8, "Automated Exploratory Data Analysis Report", 
             ln=True, align="C")
    pdf.cell(0, 8, f"Generated: {datetime.now().strftime('%B %d, %Y')}", 
             ln=True, align="C")
    pdf.ln(8)

    # Section 1: Dataset Profile 
    pdf.section_title("1. Dataset Profile")
    pdf.body_text(data_profile)

    # Section 2: AI Insights
    pdf.add_page()
    pdf.section_title("2. AI-Generated EDA Insights")
    pdf.body_text(llm_insights)

    # Section 3: Visualizations
    pdf.add_page()
    pdf.section_title("3. Auto-Generated Visualizations")
    pdf.ln(2)

    for i, chart_path in enumerate(chart_paths):
        if os.path.exists(chart_path):
            chart_name = os.path.basename(chart_path)\
                           .replace(".png", "")\
                           .replace("_", " ")\
                           .title()

            pdf.set_font("Arial", "B", 10)
            pdf.cell(0, 7, f"Chart {i+1}: {chart_name}", ln=True)

            # Add chart image
            pdf.image(chart_path, x=15, w=175)
            pdf.ln(4)

            # New page every 2 charts to avoid overflow
            if (i + 1) % 2 == 0 and i + 1 < len(chart_paths):
                pdf.add_page()

    # Section 4: Hypotheses
    pdf.add_page()
    pdf.section_title("4. AI-Generated Business Hypotheses")
    pdf.body_text(hypotheses)

    output_path = f"/kaggle/working/{dataset_name}_EDA_Report.pdf"
    pdf.output(output_path)

    return output_path

print("Generating PDF report...")

pdf_path = generate_pdf_report(
    dataset_name   = "Titanic",
    data_profile   = data_profile,
    llm_insights   = llm_insights,
    hypotheses     = hypotheses,
    chart_paths    = chart_paths
)

file_size = os.path.getsize(pdf_path) / 1024  # KB
print(f"PDF Report generated successfully!")
print(f"Location : {pdf_path}")
print(f"File Size: {file_size:.1f} KB")

Generating PDF report...


/tmp/ipykernel_57/3622685769.py:9: DeprecationWarning:

Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed

/tmp/ipykernel_57/3622685769.py:12: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_57/3622685769.py:57: DeprecationWarning:

Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed

/tmp/ipykernel_57/3622685769.py:60: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_57/3622685769.py:61: DeprecationWarning:

Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed

/tmp/ipykernel_57/3622685769.py:63: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipyker

PDF Report generated successfully!
Location : /kaggle/working/Titanic_EDA_Report.pdf
File Size: 186.0 KB


# Test EDA Agent on House Prices Dataset

In [11]:

print("   TESTING EDA AGENT ON HOUSE PRICES DATASET")

# Step 1: Load New Dataset 
df2 = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
print(f"\n Dataset loaded!")
print(f"   Rows    : {df2.shape[0]}")
print(f"   Columns : {df2.shape[1]}")

# Profile It
print("\nProfiling dataset...")
data_profile2 = profile_dataframe(df2)
print(data_profile2[:2000])
print("\n... [profile continues] ...")

# LLM Insights
print("\n Generating LLM insights...")
llm_insights2 = generate_llm_insights(
    data_profile  = data_profile2,
    dataset_name  = "House Prices"
)
print("\n--- LLM INSIGHTS ---")
print(llm_insights2)

# Generate Visualizations 
print("\n Generating visualizations...")

# House prices has 79 columns — we'll focus on key ones only
key_columns = ["SalePrice", "GrLivArea", "OverallQual", 
               "YearBuilt", "TotalBsmtSF", "GarageCars",
               "Neighborhood"]

df2_viz = df2[key_columns].copy()

# Save to temp and generate charts
os.makedirs("/kaggle/working/charts_house", exist_ok=True)
chart_paths2 = []

for col in key_columns[:-1]:  # Skip Neighborhood (categorical)
    fig = px.histogram(
        df2_viz, 
        x=col,
        title=f"Distribution of {col}",
        color_discrete_sequence=["#636EFA"]
    )
    fig.update_layout(template="plotly_white")
    path = f"/kaggle/working/charts_house/hist_{col}.png"
    fig.write_image(path)
    chart_paths2.append(path)
    print(f"    Saved: hist_{col}.png")

# Scatter: GrLivArea vs SalePrice (classic relationship)
fig = px.scatter(
    df2, 
    x="GrLivArea", 
    y="SalePrice",
    title="Living Area vs Sale Price",
    color="OverallQual",
    color_continuous_scale="Viridis",
    labels={"GrLivArea": "Above Ground Living Area (sqft)",
            "SalePrice": "Sale Price ($)"}
)
fig.update_layout(template="plotly_white")
path = "/kaggle/working/charts_house/scatter_area_price.png"
fig.write_image(path)
chart_paths2.append(path)
print("    Saved: scatter_area_price.png")

# Correlation heatmap for key numeric columns
key_numeric = ["SalePrice", "GrLivArea", "OverallQual",
               "YearBuilt", "TotalBsmtSF", "GarageCars",
               "FullBath", "TotRmsAbvGrd"]
corr2 = df2[key_numeric].corr().round(2)
fig = px.imshow(
    corr2,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Correlation Heatmap — House Price Features"
)
fig.update_layout(template="plotly_white")
path = "/kaggle/working/charts_house/correlation_heatmap.png"
fig.write_image(path)
chart_paths2.append(path)
print("    Saved: correlation_heatmap.png")

print(f"\n Total charts: {len(chart_paths2)}")

# Generate Hypotheses
print("\n Generating hypotheses...")
hypotheses2 = generate_hypotheses_v2(
    data_profile = data_profile2,
    llm_insights = llm_insights2,
    dataset_name = "House Prices"
)
print("\n--- HYPOTHESES ---")
print(hypotheses2)

# Generate PDF
print("\n Generating PDF report...")
pdf_path2 = generate_pdf_report(
    dataset_name = "HousePrices",
    data_profile = data_profile2,
    llm_insights = llm_insights2,
    hypotheses   = hypotheses2,
    chart_paths  = chart_paths2
)

file_size2 = os.path.getsize(pdf_path2) / 1024
print(f"\nPDF Report generated!")
print(f"Location  : {pdf_path2}")
print(f"File Size : {file_size2:.1f} KB")
print("\n" + "=" * 60)
print("   AGENT WORKS ON HOUSE PRICES DATASET!")
print("=" * 60)

   TESTING EDA AGENT ON HOUSE PRICES DATASET

 Dataset loaded!
   Rows    : 1460
   Columns : 81

Profiling dataset...
DATASET OVERVIEW
Total Rows    : 1460
Total Columns : 81
Column Names  : ['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'Ga

/tmp/ipykernel_57/3622685769.py:9: DeprecationWarning:

Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed

/tmp/ipykernel_57/3622685769.py:12: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_57/3622685769.py:57: DeprecationWarning:

Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed

/tmp/ipykernel_57/3622685769.py:60: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipykernel_57/3622685769.py:61: DeprecationWarning:

Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed

/tmp/ipykernel_57/3622685769.py:63: DeprecationWarning:

The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

/tmp/ipyker


PDF Report generated!
Location  : /kaggle/working/HousePrices_EDA_Report.pdf
File Size : 195.0 KB

   AGENT WORKS ON HOUSE PRICES DATASET!


In [12]:
# ============================================================
# FINAL CELL — Download All Outputs as ZIP
# ============================================================

import zipfile
import os

# ── Create zip file ─────────────────────────────────────────
zip_path = "/kaggle/working/EDA_Agent_Complete.zip"

with zipfile.ZipFile(zip_path, "w", 
                     zipfile.ZIP_DEFLATED) as zipf:

    # ── 1. PDF Reports ───────────────────────────────────────
    pdfs = [
        "/kaggle/working/Titanic_EDA_Report.pdf",
        "/kaggle/working/HousePrices_EDA_Report.pdf"
    ]
    for pdf in pdfs:
        if os.path.exists(pdf):
            zipf.write(pdf, 
                f"PDF_Reports/{os.path.basename(pdf)}")
            print(f"✅ Added: {os.path.basename(pdf)}")

    # ── 2. Titanic Charts ────────────────────────────────────
    titanic_charts = "/kaggle/working/charts"
    if os.path.exists(titanic_charts):
        for f in os.listdir(titanic_charts):
            if f.endswith(".png"):
                zipf.write(
                    os.path.join(titanic_charts, f),
                    f"Charts/Titanic/{f}"
                )
                print(f"✅ Added: Titanic/{f}")

    # ── 3. House Prices Charts ───────────────────────────────
    house_charts = "/kaggle/working/charts_house"
    if os.path.exists(house_charts):
        for f in os.listdir(house_charts):
            if f.endswith(".png"):
                zipf.write(
                    os.path.join(house_charts, f),
                    f"Charts/HousePrices/{f}"
                )
                print(f"✅ Added: HousePrices/{f}")

# ── Show final size ──────────────────────────────────────────
size = os.path.getsize(zip_path) / (1024*1024)
print(f"\n{'='*50}")
print(f"✅ ZIP Created Successfully!")
print(f"📁 Location : {zip_path}")
print(f"📦 File Size: {size:.2f} MB")
print(f"{'='*50}")
print("\n👉 Go to Output tab → Download EDA_Agent_Complete.zip")

✅ Added: Titanic_EDA_Report.pdf
✅ Added: HousePrices_EDA_Report.pdf
✅ Added: Titanic/hist_SibSp.png
✅ Added: Titanic/bar_Sex.png
✅ Added: Titanic/bar_Embarked.png
✅ Added: Titanic/hist_Age.png
✅ Added: Titanic/hist_Fare.png
✅ Added: Titanic/hist_Survived.png
✅ Added: Titanic/hist_Parch.png
✅ Added: Titanic/correlation_heatmap.png
✅ Added: Titanic/hist_Pclass.png
✅ Added: Titanic/survival_overview.png
✅ Added: HousePrices/scatter_area_price.png
✅ Added: HousePrices/hist_GarageCars.png
✅ Added: HousePrices/hist_GrLivArea.png
✅ Added: HousePrices/hist_YearBuilt.png
✅ Added: HousePrices/hist_TotalBsmtSF.png
✅ Added: HousePrices/hist_OverallQual.png
✅ Added: HousePrices/correlation_heatmap.png
✅ Added: HousePrices/hist_SalePrice.png

✅ ZIP Created Successfully!
📁 Location : /kaggle/working/EDA_Agent_Complete.zip
📦 File Size: 0.72 MB

👉 Go to Output tab → Download EDA_Agent_Complete.zip
